# 107 · Superposition: do two objects combine linearly?
**Source of truth for superposition** (supersedes the additivity section of `src8/EXPERIMENTS.md`). Data: `src8/superposition.py` → `runs/superposition/{pairs,model}.csv`; `src8/additivity.py` → `runs/additivity_src8.csv`.

**Physics.** To first order (Rayleigh perturbation), each mode's frequency shift is a sum over the added masses, $\delta\omega_k \propto \sum_i m_i\,\phi_k(x_i)^2$. If that holds, the change two objects make to the spectrum is the sum of the changes each makes alone. Then two-object training scenes could be synthesized from single-object captures.

**Setup.** For every two-object capture (two-cubes, cube-vase; 3 boxes), each object is matched to the single-object capture of the same type and speaker whose mask overlaps it most (IoU ≥ .5).
$v$ = log power minus that of the nearest-in-time empty box of the same speaker, shape (100 lasers, 1170 bins, 2 axes): $v_1, v_2$ for the singles, $v_3$ for the two-object capture.
Maps are fit on the train pairs and scored on the **held-out pairs** (the eval split's positions; 214 of 1375).

| model | prediction of $v_3$ | parameters |
|---|---|---|
| sum | $v_1+v_2$ | 0 |
| mean | $b$ (the average two-object deviation) | d |
| scalar | $a(v_1+v_2)+b$ | 1 + d |
| diag | $a_i(v_1+v_2)_i+b_i$ | 2d |
| freq | $A_F$ (1170×1170) on every laser's spectrum, $+b$ | 1.4 M |
| laser | $A_L$ (100×100) on every bin's laser field, $+b$ | 10 k |
| lowrank | $b + U_3 M U_s^\top (v_1+v_2)$, rank r | r² |
| one | $a\,v_{big}+b$: the bigger object alone | 1 + d |
| **inter** | diag $+\,c_i\,(v_1 \odot v_2)_i$: an interaction term per feature | 3d |
| **rbf** | kernel ridge (gaussian kernel) on lowrank's coefficients: a smooth nonlinear map | n·r |

**The linearity test.** Least squares only says how well a linear map fits. Whether the structure *is* linear needs a nonlinear competitor given the same inputs: **inter vs diag** (do the objects interact, feature by feature?) and **rbf vs lowrank** (any smooth nonlinearity in the dominant directions?). If neither beats its linear twin on held-out pairs, the two-object capture is as linear in $v_1, v_2$ as the data can tell, and what the linear map misses is noise or session drift, not interaction. (Checked on toy data: on an interacting truth $v_1+v_2+\tfrac12 v_1 v_2$, inter reaches fve .995 vs diag .879; on a linear truth both give .989.)

**Is it just least squares?** Yes. $A v_1 + A v_2 = A(v_1+v_2)$, so fitting $A$ is linear regression of $v_3$ on $s = v_1+v_2$, $\min_A \sum_{pairs}\lVert v_3 - A s\rVert^2$, solved in closed form.
The catch is size. A general $A$ is $d\times d$ with $d = 100\cdot1170\cdot2 = 234$k, i.e. $5.5\times10^{10}$ unknowns against ~1100 train pairs: hopelessly underdetermined.
So $A$ needs structure or a rank limit (the rows above) plus ridge, and every fit is scored on held-out pairs (a fit on its own training pairs can always look good).
**fve** = $1-\lVert v_3-\hat v_3\rVert^2/\lVert v_3\rVert^2$ (1 = perfect, 0 = no better than predicting no change, < 0 = worse than that).
**Ceiling**: the same object at the same spot recorded twice correlates only r ≈ .54 (`r_rep`), so no map can reach fve ≈ 1.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path('..')
SP = ROOT / 'runs/superposition'
DOMAINS = [d for d in ('model', 'log_power') if (SP / d / 'pairs.csv').exists()]
DOM = os.environ.get('NB107_DOMAIN', DOMAINS[0])                 # the domain of sections 3-5
E = {d: pd.read_csv(SP / d / 'pairs.csv') for d in DOMAINS}      # held-out pairs: fve/<model>, r/<model>, and the *_shuf control
LO = {d: pd.read_csv(SP / d / 'lolo.csv') for d in DOMAINS if (SP / d / 'lolo.csv').exists()}   # leave one layout (session) out
MO = {d: pd.read_csv(SP / d / 'model.csv') for d in DOMAINS}     # trained model on real vs synthetic inputs
A = pd.read_csv(ROOT / 'runs/additivity_src8.csv')              # all pairs, incl. the repeat ceiling r_rep (log power)
MODELS = [m for m in ['sum', 'mean', 'scalar', 'diag', 'inter', 'freq', 'laser', 'lowrank', 'rbf', 'one'] if f'fve/{m}' in E[DOM]]
BOXES = ['gastro', 'plastic', 'cardboard']
COL = dict(zip(['sum', 'mean', 'scalar', 'diag', 'inter', 'freq', 'laser', 'lowrank', 'rbf', 'one'], ['#7a7a7a', '#c8a2c8', '#2a78d6', '#1baf7a', '#0e6b49', '#eb6834', '#e34948', '#8a5cd6', '#4b2d8a', '#b0b0b0']))
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3})
BINS, LAB = [0, .25, .4, .55, 1.5], ['<.25', '.25-.4', '.4-.55', '>.55']
for d in DOMAINS:
    E[d]['dist_bin'] = pd.cut(E[d].dist, BINS, labels=LAB); MO[d]['dist_bin'] = pd.cut(MO[d].dist, BINS, labels=LAB)
print('domains', DOMAINS, '| sections 3-5 use', DOM)
print(len(E[DOM]), 'held-out pairs'); print(pd.crosstab(E[DOM].box, E[DOM].types).to_string())

## 1 The pairs: how far apart, which types

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 3.6), sharey=True)
for a, b in zip(ax, BOXES):
    for ty, c in (('cube+cube', '#2a78d6'), ('cube+vase', '#eb6834')):
        x = E[DOM][(E[DOM].box == b) & (E[DOM].types == ty)].dist
        a.hist(x, bins=np.linspace(0, 1, 21), alpha=.6, color=c, label=f'{ty} ({len(x)})')
    a.set_title(b); a.set_xlabel('centroid distance (box fractions)'); a.legend()
ax[0].set_ylabel('held-out pairs'); plt.tight_layout()

## 2 Which linear map predicts the two-object capture?
Three tests per domain, fve per box:
- **held-out positions**: fit on the train pairs, score on the eval split's pairs (other positions, *same* recording sessions);
- **shuffled control**: the same fitted maps, fed v1 + v2 from *another* pair of the same session. A map that only learned the session's drift keeps its score; one that reads these objects collapses;
- **leave one session out**: fit on every other layout, score on the held-out layout (its own drift unseen): the honest number for synthesizing new scenes.

In [ ]:
def bars(ax, T, title, models=MODELS):
    x, w = np.arange(len(T)), .8 / len(models)
    for i, m in enumerate(models): ax.bar(x + i * w, T[m], w, color=COL[m], label=m)
    ax.set_xticks(x + .4 - w / 2, T.index); ax.set_title(title); ax.axhline(0, color='k', lw=.8); ax.set_ylim(-1.5, .9)
def by_box(df, pre):
    T = df.groupby('box')[[f'{pre}/{m}' for m in MODELS]].mean().reindex(BOXES); T.loc['all'] = df[[f'{pre}/{m}' for m in MODELS]].mean()
    return T.rename(columns=lambda c: c.split('/')[1])
for d in DOMAINS:
    fig, ax = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
    bars(ax[0], by_box(E[d], 'fve'), f'{d}: held-out positions')
    bars(ax[1], by_box(E[d], 'fve_shuf'), f'{d}: shuffled control (other pair, same session)')
    if d in LO: bars(ax[2], by_box(LO[d], 'fve'), f'{d}: leave one session out')
    ax[0].set_ylabel('fve (clipped at -1.5)'); ax[-1].legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()
    display(pd.concat({'held-out': by_box(E[d], 'fve'), 'shuffled': by_box(E[d], 'fve_shuf'), **({'session-out': by_box(LO[d], 'fve')} if d in LO else {})}, axis=1).round(2))
rep = A.groupby('box')['log_power/r_rep'].mean().reindex(BOXES); rep.loc['all'] = A['log_power/r_rep'].mean()
print('repeat ceiling r_rep (log power, same object same spot recorded twice):', rep.round(2).to_dict())

## 2b Is it linear? Each nonlinear model minus its linear twin, per held-out pair
Bars: mean fve gain with a 95% bootstrap interval (2000 resamples of pairs). Above 0 means the nonlinear model explains something the linear one can't.

In [ ]:
def boot(x, n=2000, seed=0):
    x = np.asarray(x); r = np.random.default_rng(seed)
    m = x[r.integers(0, len(x), (n, len(x)))].mean(1); return x.mean(), *np.percentile(m, [2.5, 97.5])
PAIRS = [(a, b) for a, b in (('inter', 'diag'), ('rbf', 'lowrank')) if f'fve/{a}' in E[DOM]]
for name, src in (('held-out positions', E), ('leave one session out', LO)):
    fig, ax = plt.subplots(1, len(DOMAINS), figsize=(18, 3.6), squeeze=False)
    rows = []
    for a_, d in zip(ax[0], DOMAINS):
        if d not in src: continue
        df = src[d]
        for k, (nl, li) in enumerate(PAIRS):
            for j, b in enumerate(BOXES + ['all']):
                x = (df if b == 'all' else df[df.box == b])
                m, lo, hi = boot(x[f'fve/{nl}'] - x[f'fve/{li}'])
                a_.bar(j + k * .4, m, .38, color=COL[nl], label=f'{nl} - {li}' if j == 0 else None)
                a_.plot([j + k * .4] * 2, [lo, hi], color='k', lw=1.2)
                rows.append(dict(domain=d, test=f'{nl} - {li}', box=b, gain=m, lo=lo, hi=hi))
        a_.set_xticks(np.arange(4) + .2, BOXES + ['all']); a_.axhline(0, color='k', lw=.8); a_.set_title(f'{d}: {name}'); a_.legend(fontsize=8)
    ax[0, 0].set_ylabel('fve gain over the linear twin'); plt.tight_layout(); plt.show()
    display(pd.DataFrame(rows).pivot_table(index=['domain', 'test'], columns='box', values='gain').round(3))

## 3 Does it matter how far apart the objects are?

In [ ]:
SHOW = [m for m in ['sum', 'mean', 'scalar', 'freq', 'lowrank', 'one'] if m in MODELS]
for df, title in ((E[DOM], 'held-out positions'), (LO.get(DOM), 'leave one session out')):
    if df is None: continue
    df = df.assign(dist_bin=pd.cut(df.dist, BINS, labels=LAB))
    fig, ax = plt.subplots(1, 3, figsize=(18, 3.6), sharey=True)
    for a, b in zip(ax, BOXES):
        g = df[df.box == b].groupby('dist_bin', observed=False)
        for m in SHOW: a.plot(LAB, g[f'r/{m}'].mean(), 'o-', color=COL[m], lw=2, label=m)
        a.set_title(f'{b} ({title}; n = {", ".join(map(str, g.size()))})'); a.set_xlabel('centroid distance (box fractions)')
    ax[0].set_ylabel(f'correlation with the real v3 ({DOM})'); ax[-1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 4 Same type (cube+cube) vs different (cube+vase)

In [ ]:
for df, title in ((E[DOM], 'held-out positions'), (LO.get(DOM), 'leave one session out')):
    if df is None: continue
    G = df.groupby(['box', 'types'])[[f'r/{m}' for m in MODELS]].mean()
    fig, ax = plt.subplots(1, 3, figsize=(18, 3.6), sharey=True)
    for a, b in zip(ax, BOXES):
        g = G.loc[b]; x, w = np.arange(len(g)), .8 / len(MODELS)
        for i, m in enumerate(MODELS): a.bar(x + i * w, g[f'r/{m}'], w, color=COL[m], label=m)
        a.set_xticks(x + .4 - w / 2, g.index); a.set_title(f'{b}: r with the real v3 ({title})')
    ax[-1].legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()
    display(G.round(2))

## 5 The real test: what does the trained model draw from a synthetic scene?
s8-best-mag on the held-out pairs: iou@0.08 against the union of both masks, and *both found* (each object > 10% covered). Inputs:
- **real**: the real capture;
- **sum**: the plain sum;
- **best**: the best linear fit;
- **mean**: the mean two-object deviation alone, which carries no information about these objects;
- **one**: one object alone;
- **proj**: the *real* capture kept only in the low-rank map's output subspace. If the model fails on it too, that subspace lacks what the model reads, however much variance it explains.

In [ ]:
VC = dict(real='#222222', sum='#7a7a7a', best='#2a78d6', mean='#c8a2c8', one='#b0b0b0', proj='#8a5cd6')
for d in DOMAINS:
    M_ = MO[d]; V = [v for v in VC if f'iou/{v}' in M_]
    fig, ax = plt.subplots(1, 2, figsize=(18, 4))
    for k, (met, t) in enumerate((('iou', 'iou@0.08 vs the union'), ('both', 'both objects found'))):
        g = M_.groupby('box')[[f'{met}/{v}' for v in V]].mean().reindex(BOXES); g.loc['all'] = M_[[f'{met}/{v}' for v in V]].mean()
        x, ww = np.arange(len(g)), .8 / len(V)
        for i, v in enumerate(V): ax[k].bar(x + i * ww, g[f'{met}/{v}'], ww, color=VC[v], label=v)
        ax[k].set_xticks(x + .4 - ww / 2, g.index); ax[k].set_title(f'{d}: {t}')
    ax[0].legend(); plt.tight_layout(); plt.show()
    display(M_.groupby(['box', 'types'])[[f'iou/{v}' for v in V] + [f'both/{v}' for v in V]].mean().round(2))
    display(M_.groupby('dist_bin', observed=False)[[f'iou/{v}' for v in V] + [f'both/{v}' for v in V]].mean().round(2))

## 6 Verdict
_filled in after the run_